# DepthDif public inference showcase

This Colab-style notebook installs `depth-recon` from PyPI, lets you choose an ISO year/week, lets you draw a lon/lat rectangle, runs the public DepthDif API, and displays the prediction on a Leaflet map.

The public path downloads the Hugging Face checkpoint/configs/land mask, then downloads the matching EN4/ARGO profiles and OSTIA SST field for the selected ISO week. GLORYS is not required for this standard inference flow.

In [ ]:
# @title 1. Install DepthDif and notebook widgets { display-mode: "form" }
%pip install -q depth-recon copernicusmarine ipywidgets ipyleaflet


In [ ]:
# @title 2. Hidden notebook helpers { display-mode: "form" }
from __future__ import annotations

import base64
from datetime import date
from io import BytesIO
import json
import os
from pathlib import Path

from IPython.display import HTML, display
from ipyleaflet import DrawControl, GeoJSON, ImageOverlay, LayersControl, Map, basemaps
import ipywidgets as widgets
from matplotlib import colormaps
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
import yaml

from inference.api import resolve_hf_assets, run_week_inference

try:
    from google.colab import output

    # Colab needs this for third-party ipywidgets such as ipyleaflet.
    output.enable_custom_widget_manager()
except Exception:
    pass

STATE = {
    'hf_repo_id': 'donike/depthdif',
    'hf_revision': 'main',
    'cache_dir': Path('/content/depthdif_cache'),
    'output_root': Path('/content/depthdif_outputs'),
    'device': 'cuda',
    'batch_size': 2,
    'force_download': False,
    'rectangle': None,
    'assets': None,
    'run_dir': None,
}


def _html_status(message: str, kind: str = 'info') -> HTML:
    """Return a compact notebook status message."""
    colors = {'info': '#334155', 'ok': '#166534', 'warn': '#92400e'}
    color = colors.get(kind, colors['info'])
    return HTML(f'<span style="color:{color}">{message}</span>')


def iso_weeks_in_year(year: int) -> list[int]:
    """Return the valid ISO week numbers for one calendar year."""
    return list(range(1, date(int(year), 12, 28).isocalendar().week + 1))


def _runtime_widgets() -> dict[str, widgets.Widget]:
    """Create or return the runtime configuration widgets."""
    if 'runtime_widgets' not in STATE:
        STATE['runtime_widgets'] = {
            'hf_repo_id': widgets.Text(
                value=str(STATE['hf_repo_id']), description='HF repo', layout=widgets.Layout(width='520px')
            ),
            'hf_revision': widgets.Text(
                value=str(STATE['hf_revision']), description='Revision', layout=widgets.Layout(width='260px')
            ),
            'cache_dir': widgets.Text(
                value=str(STATE['cache_dir']), description='Cache', layout=widgets.Layout(width='520px')
            ),
            'output_root': widgets.Text(
                value=str(STATE['output_root']), description='Outputs', layout=widgets.Layout(width='520px')
            ),
            'device': widgets.Dropdown(options=['cuda', 'cpu', 'auto'], value=str(STATE['device']), description='Device'),
            'batch_size': widgets.BoundedIntText(value=int(STATE['batch_size']), min=0, max=128, description='Batch'),
            'force_download': widgets.Checkbox(value=bool(STATE['force_download']), description='Force downloads'),
        }
    return STATE['runtime_widgets']


def _sync_runtime_settings() -> None:
    """Copy current runtime widget values into STATE."""
    runtime = _runtime_widgets()
    STATE['hf_repo_id'] = runtime['hf_repo_id'].value.strip()
    STATE['hf_revision'] = runtime['hf_revision'].value.strip()
    STATE['cache_dir'] = Path(runtime['cache_dir'].value).expanduser()
    STATE['output_root'] = Path(runtime['output_root'].value).expanduser()
    STATE['device'] = runtime['device'].value
    STATE['batch_size'] = int(runtime['batch_size'].value)
    STATE['force_download'] = bool(runtime['force_download'].value)


def configure_depthdif_runtime() -> None:
    """Display runtime options that are safe for most Colab users to change."""
    runtime = _runtime_widgets()
    for widget in runtime.values():
        widget.observe(lambda change: _sync_runtime_settings(), names='value')
    _sync_runtime_settings()
    display(widgets.VBox([
        widgets.HBox([runtime['hf_repo_id'], runtime['hf_revision']]),
        runtime['cache_dir'],
        runtime['output_root'],
        widgets.HBox([runtime['device'], runtime['batch_size'], runtime['force_download']]),
    ]))


def _auth_widgets() -> dict[str, widgets.Widget]:
    """Create or return Copernicus Marine credential widgets."""
    if 'auth_widgets' not in STATE:
        STATE['auth_widgets'] = {
            'username': widgets.Text(
                value=os.environ.get('COPERNICUSMARINE_SERVICE_USERNAME', ''),
                description='Username',
                layout=widgets.Layout(width='420px'),
            ),
            'token': widgets.Password(
                value=os.environ.get('COPERNICUSMARINE_SERVICE_PASSWORD', ''),
                description='Token/key',
                layout=widgets.Layout(width='420px'),
            ),
            'button': widgets.Button(description='Use credentials', icon='key', button_style='primary'),
            'status': widgets.HTML(''),
        }
    return STATE['auth_widgets']


def _sync_copernicus_environment(required: bool = False) -> tuple[str, str]:
    """Store Copernicus credentials in environment variables for the current runtime."""
    auth = _auth_widgets()
    username = auth['username'].value.strip()
    token = auth['token'].value
    if required and (not username or not token):
        raise ValueError('Enter your Copernicus Marine username and token/key before inference.')
    if username:
        os.environ['COPERNICUSMARINE_SERVICE_USERNAME'] = username
    if token:
        # The toolbox reads the API key/token from its password environment variable.
        os.environ['COPERNICUSMARINE_SERVICE_PASSWORD'] = token
    return username, token


def authenticate_copernicusmarine() -> None:
    """Open Copernicus Marine username and token inputs for OSTIA downloads."""
    auth = _auth_widgets()

    def handle_click(_button: widgets.Button) -> None:
        _sync_copernicus_environment(required=True)
        auth['status'].value = '<span style="color:#166534">Credentials loaded for this Colab runtime.</span>'

    auth['button'].on_click(handle_click)
    display(widgets.VBox([
        widgets.HTML('Paste the Copernicus Marine API key/token into the password box. It is kept only in this runtime.'),
        auth['username'],
        auth['token'],
        widgets.HBox([auth['button'], auth['status']]),
    ]))


def rectangle_from_geojson(geo_json: dict) -> tuple[float, float, float, float]:
    """Extract lon_min, lat_min, lon_max, lat_max from a drawn rectangle."""
    geometry = geo_json.get('geometry', geo_json)
    if geometry.get('type') == 'FeatureCollection':
        geometry = geometry['features'][0]['geometry']
    coords = geometry['coordinates'][0]
    lons = [float(point[0]) for point in coords]
    lats = [float(point[1]) for point in coords]
    return min(lons), min(lats), max(lons), max(lats)


def select_week_and_bbox() -> None:
    """Display ISO week controls and a Leaflet rectangle picker."""
    year_options = list(range(2010, date.today().year + 1))
    year_widget = widgets.Dropdown(options=year_options, value=2015, description='Year')
    week_widget = widgets.Dropdown(options=iso_weeks_in_year(2015), value=25, description='ISO week')
    rectangle_status = widgets.HTML('Draw one rectangle on the map.')
    draw_map = Map(
        center=(20.0, 0.0),
        zoom=2,
        basemap=basemaps.Esri.WorldImagery,
        scroll_wheel_zoom=True,
    )
    draw_control = DrawControl(
        rectangle={'shapeOptions': {'color': '#0f766e', 'fillOpacity': 0.12}},
        polygon={},
        polyline={},
        circle={},
        circlemarker={},
        marker={},
    )

    def sync_week_options(change: dict) -> None:
        """Keep the ISO-week dropdown valid when the year changes."""
        weeks = iso_weeks_in_year(int(change['new']))
        current_week = int(week_widget.value)
        week_widget.options = weeks
        week_widget.value = current_week if current_week in weeks else weeks[-1]

    def handle_draw(target: DrawControl, action: str, geo_json: dict) -> None:
        """Store the latest rectangle drawn in the Leaflet widget."""
        if action in {'created', 'edited'}:
            bounds = rectangle_from_geojson(geo_json)
            STATE['rectangle'] = bounds
            rectangle_status.value = (
                'Selected rectangle: '
                f'lon {bounds[0]:.3f} to {bounds[2]:.3f}, '
                f'lat {bounds[1]:.3f} to {bounds[3]:.3f}'
            )
        elif action == 'deleted':
            STATE['rectangle'] = None
            rectangle_status.value = 'Draw one rectangle on the map.'

    year_widget.observe(sync_week_options, names='value')
    draw_control.on_draw(handle_draw)
    draw_map.add_control(draw_control)
    STATE['year_widget'] = year_widget
    STATE['week_widget'] = week_widget
    display(widgets.HBox([year_widget, week_widget]), rectangle_status, draw_map)


def resolve_depthdif_assets() -> None:
    """Resolve the Hugging Face config files and checkpoint used for inference."""
    _sync_runtime_settings()
    assets = resolve_hf_assets(
        config_repo=STATE['hf_repo_id'],
        revision=STATE['hf_revision'],
        cache_dir=STATE['cache_dir'],
        force_download=STATE['force_download'],
    )
    STATE['assets'] = assets
    display(_html_status('Model assets resolved.', 'ok'))
    display(HTML(
        '<br>'.join([
            f'Model config: {assets.model_config}',
            f'Data config: {assets.data_config}',
            f'Train config: {assets.train_config}',
            f'Checkpoint: {assets.checkpoint}',
        ])
    ))


def prediction_tif_from_run(run_dir: Path, suffix: str = 'surface') -> Path:
    """Resolve one exported prediction GeoTIFF from a DepthDif run directory."""
    run_dir = Path(run_dir)
    summary_path = run_dir / 'run_summary.yaml'
    if summary_path.exists():
        summary = yaml.safe_load(summary_path.read_text()) or {}
        for record in summary.get('depth_exports', []):
            label = str(record.get('label', '')).lower()
            record_suffix = str(record.get('suffix', '')).lower()
            if suffix.lower() in {label, record_suffix}:
                path = Path(record['prediction_tif_path'])
                return path if path.is_absolute() else run_dir / path
        if summary.get('prediction_tif_path'):
            path = Path(summary['prediction_tif_path'])
            return path if path.is_absolute() else run_dir / path
    matches = sorted(run_dir.glob(f'*_prediction_{suffix}.tif'))
    if not matches:
        matches = sorted(run_dir.glob('*_prediction*.tif'))
    if not matches:
        raise FileNotFoundError(f'No prediction GeoTIFF found in {run_dir}.')
    return matches[0]


def geotiff_leaflet_payload(
    tif_path: Path,
    *,
    cmap_name: str = 'turbo',
    percentile_clip: tuple[float, float] = (2.0, 98.0),
) -> tuple[str, tuple[tuple[float, float], tuple[float, float]]]:
    """Convert a single-band GeoTIFF into a Leaflet image data URI and bounds."""
    with rasterio.open(tif_path) as src:
        data = src.read(1, masked=True).astype('float32')
        if src.crs is None:
            west, south, east, north = src.bounds
        else:
            west, south, east, north = transform_bounds(src.crs, 'EPSG:4326', *src.bounds, densify_pts=21)

    values = np.ma.filled(data, np.nan)
    valid = np.isfinite(values)
    if not valid.any():
        raise ValueError(f'Prediction raster has no finite values: {tif_path}')
    low, high = np.nanpercentile(values[valid], percentile_clip)
    if not np.isfinite(high - low) or high <= low:
        normalized = np.zeros_like(values, dtype='float32')
    else:
        normalized = np.clip((values - low) / (high - low), 0.0, 1.0)

    # Leaflet receives an RGBA PNG because browsers cannot render GeoTIFFs directly.
    rgba = colormaps[cmap_name](normalized)
    rgba[..., 3] = np.where(valid, 0.78, 0.0)
    buffer = BytesIO()
    plt.imsave(buffer, rgba, format='png')
    data_uri = 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')
    return data_uri, ((south, west), (north, east))


def add_geojson_layer(map_obj: Map, path: Path, name: str) -> None:
    """Add a GeoJSON layer to a Leaflet map when the file exists."""
    path = Path(path)
    if path.exists():
        map_obj.add_layer(GeoJSON(data=json.loads(path.read_text()), name=name))


def build_result_map(run_dir: Path, rectangle: tuple[float, float, float, float], *, depth_suffix: str = 'surface') -> Map:
    """Build a Leaflet result map with the prediction overlay and ARGO points."""
    tif_path = prediction_tif_from_run(run_dir, suffix=depth_suffix)
    data_uri, bounds = geotiff_leaflet_payload(tif_path)
    (south, west), (north, east) = bounds
    center = ((south + north) / 2.0, (west + east) / 2.0)
    result_map = Map(center=center, zoom=4, basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True)
    result_map.add_layer(ImageOverlay(url=data_uri, bounds=bounds, name=f'DepthDif {depth_suffix}'))
    add_geojson_layer(result_map, next(iter(sorted(Path(run_dir).glob('*_argo_points.geojson'))), Path('__missing__')), 'ARGO observations')
    add_geojson_layer(result_map, next(iter(sorted(Path(run_dir).glob('*_patches.geojson'))), Path('__missing__')), 'Selected patches')
    result_map.add_control(LayersControl(position='topright'))
    return result_map


def run_depthdif_inference() -> None:
    """Run public API inference with automatic ARGO and OSTIA downloads."""
    _sync_runtime_settings()
    _sync_copernicus_environment(required=True)
    if STATE.get('rectangle') is None:
        raise ValueError('Draw a rectangle before running inference.')
    if 'year_widget' not in STATE or 'week_widget' not in STATE:
        raise ValueError('Run select_week_and_bbox() before inference.')
    if STATE.get('assets') is None:
        resolve_depthdif_assets()

    batch_size = None if int(STATE['batch_size']) <= 0 else int(STATE['batch_size'])
    run_dir = run_week_inference(
        year=int(STATE['year_widget'].value),
        iso_week=int(STATE['week_widget'].value),
        rectangle=STATE['rectangle'],
        output_root=STATE['output_root'],
        device=STATE['device'],
        checkpoint=STATE['assets'].checkpoint,
        config_repo=STATE['hf_repo_id'],
        revision=STATE['hf_revision'],
        cache_dir=STATE['cache_dir'],
        auto_download_argo=True,
        auto_download_ostia=True,
        batch_size=batch_size,
        force_download=STATE['force_download'],
    )
    STATE['run_dir'] = Path(run_dir)
    display(_html_status(f'Run directory: {STATE["run_dir"]}', 'ok'))


def display_depthdif_result(depth_suffix: str = 'surface') -> None:
    """Display the exported DepthDif prediction on a Leaflet map."""
    if STATE.get('run_dir') is None:
        raise ValueError('Run run_depthdif_inference() first.')
    result_map = build_result_map(STATE['run_dir'], STATE['rectangle'], depth_suffix=depth_suffix)
    display(result_map)


Run the remaining cells from top to bottom. The visible cells are thin wrappers; the implementation is kept in hidden Colab form cells.

In [ ]:
# @title 3. Runtime options { display-mode: "form" }
configure_depthdif_runtime()


In [ ]:
# @title 4. Copernicus Marine authentication { display-mode: "form" }
authenticate_copernicusmarine()


In [ ]:
# @title 5. Select week and draw bbox { display-mode: "form" }
select_week_and_bbox()


In [ ]:
# @title 6. Resolve model assets { display-mode: "form" }
resolve_depthdif_assets()


In [ ]:
# @title 7. Run DepthDif inference { display-mode: "form" }
run_depthdif_inference()


In [ ]:
# @title 8. Display prediction map { display-mode: "form" }
display_depthdif_result()
